# Pipeline Evidence — DeBERTa Training

Setup → Load evidence → Fine-tune on BM25-retrieved evidence → Evaluate on test set.

**Before running:** change `BRANCH` in Cell 1 to `"vita/synmap_pipeline_refined"`, and upload `cache/` folder to `MyDrive/Final_Assignment/cache/`.

In [ ]:
import subprocess, os, sys

# ── BRANCH / VARIANT pairing ──────────────────────────────────────────────
BRANCH = "vita/synmap_pipeline_refined"   # ← must match VARIANT in cell 7

REPO_DIR = "/content/comp90042"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", BRANCH,
         "https://github.com/VitaChien/COMP90042_2026.git", REPO_DIR],
        check=True
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Code source: {REPO_DIR}  branch={BRANCH}")

# ── Drive mount — artifact storage (persists across sessions) ─────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/Final_Assignment"
os.chdir(BASE_DIR)
print(f"Artifact dir: {BASE_DIR}")

In [ ]:
!pip install bm25s sentence-transformers nltk huggingface_hub spacy bitsandbytes -q
!python -m spacy download en_core_web_sm -q


In [ ]:
# ── Credentials & repo config ─────────────────────────────────────────────
# Colab Secrets setup: open the key icon (🔑) in the left Colab sidebar and add:
#   HF_TOKEN      — from huggingface.co/settings/tokens  (write access)
#
# If you see "Transport endpoint is not connected": Runtime → Reconnect first.
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN     = userdata.get("HF_TOKEN")
HF_USERNAME  = "ColeH0415"

# ── Branch identity ───────────────────────────────────────────────────────
# Jerry/pipeline        → "pipeline"
# Jerry/SynMap_pipeline → "synmap"
VARIANT = "synmap"

# ── HF Hub repos — variant-specific so branches don't overwrite each other ─
HF_DEBERTA_REPO = f"{HF_USERNAME}/comp90042-deberta-{VARIANT}"
HF_CE_REPO      = f"{HF_USERNAME}/comp90042-ce-{VARIANT}"
HF_BM25_REPO    = f"{HF_USERNAME}/comp90042-bm25-{VARIANT}"

# ── Absolute Drive paths ──────────────────────────────────────────────────
# Shared: raw data and dense embeddings (identical across branches)
DENSE_CACHE_PATH = os.path.join(BASE_DIR, "dense_embeddings.npy")
# Branch-specific: all artifacts that touch BM25 get a VARIANT suffix
BM25_CACHE_DIR   = os.path.join(BASE_DIR, f"bm25_index_{VARIANT}")
CE_FINETUNED_DIR = os.path.join(BASE_DIR, f"ce-finetuned-{VARIANT}")
CE_PAIRS_PATH    = os.path.join(BASE_DIR, f"ce_train_pairs_{VARIANT}.pkl")
DEBERTA_BEST_DIR = os.path.join(BASE_DIR, f"deberta-best-{VARIANT}")
DEV_PRED_PATH    = os.path.join(BASE_DIR, f"dev-predictions-{VARIANT}.json")
TEST_PRED_PATH   = os.path.join(BASE_DIR, f"predictions-{VARIANT}.json")

# ── Auth guard + login ────────────────────────────────────────────────────
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found in Colab Secrets. Open the key icon (🔑) in the "
        "left sidebar and add HF_TOKEN with write scope, then re-run this cell."
    )
if HF_USERNAME.startswith("your-hf"):
    raise RuntimeError("Set HF_USERNAME to your real HuggingFace username.")

try:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"HuggingFace Hub authenticated as {HF_USERNAME} ✓")
    print(f"VARIANT: {VARIANT}  →  artifacts suffixed with '_{VARIANT}'")
    print(f"  BM25_CACHE_DIR:   {BM25_CACHE_DIR}")
    print(f"  DEBERTA_BEST_DIR: {DEBERTA_BEST_DIR}")
    print(f"  HF_DEBERTA_REPO:  {HF_DEBERTA_REPO}")
    print(f"  HF_CE_REPO:       {HF_CE_REPO}")
    print(f"  HF_BM25_REPO:     {HF_BM25_REPO}")
except Exception as e:
    print(f"HF login failed: {e}")
    raise

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import json
from sklearn.metrics import classification_report, f1_score

# retriever.py / classifier.py are loaded from REPO_DIR (git-cloned in cell 4).

In [ ]:
from classifier import (
    train_deberta,
)
from evaluate_retriever import evaluate_retriever, evaluate_retriever_by_label
print("All modules imported successfully.")

# ── Inline fallback: run this block only if the imports above fail ────────
# (e.g. .py files not yet synced to Drive, or running without retriever.py)
# Uncomment and run this cell to exec the .py files directly:
# for _f in ["retriever.py", "classifier.py"]:
#     if os.path.exists(_f):
#         exec(open(_f).read()); print(f"Inlined: {_f}")


In [ ]:
with open(os.path.join(BASE_DIR, "data", "evidence.json")) as f:
    evidence_dict = json.load(f)

print(f"Loaded {len(evidence_dict):,} evidence passages")
for k in list(evidence_dict.keys())[:3]:
    print(f"  {k}: {evidence_dict[k][:90]}")

## 2.4b Fine-Tune on Pipeline-Retrieved Evidence

Trains (or reloads) DeBERTa using pre-cached BM25 retriever results instead of gold evidence.
For NOT_ENOUGH_INFO claims the cache already contains retrieved evidence IDs, so `bm25_retriever=None` is correct.

In [ ]:
from classifier import load_retriever_cache
from collections import Counter

CACHE_DIR = os.path.join(BASE_DIR, "cache")
DATA_DIR  = os.path.join(BASE_DIR, "data")

TRAIN_CACHE = os.path.join(CACHE_DIR, "train-retriever-only-k4-bm25200.json")
DEV_CACHE   = os.path.join(CACHE_DIR, "dev-retriever-only-k4-bm25200.json")
TEST_CACHE  = os.path.join(CACHE_DIR, "test-retriever-only-k4-bm25200.json")

# The retriever cache's `claim_label` field is unreliable - only `evidences`
# (the retriever output) is trustworthy. gold_path overwrites claim_label and
# claim_text from the gold *-claims.json files. The test set is unlabelled,
# so it keeps the cache fields as-is (its label is unused - see prediction cell).
train_pipeline = load_retriever_cache(TRAIN_CACHE, gold_path=os.path.join(DATA_DIR, "train-claims.json"))
dev_pipeline   = load_retriever_cache(DEV_CACHE,   gold_path=os.path.join(DATA_DIR, "dev-claims.json"))
test_pipeline  = load_retriever_cache(TEST_CACHE)

for name, d in [("Train", train_pipeline), ("Dev", dev_pipeline), ("Test", test_pipeline)]:
    counts = dict(Counter(v["claim_label"] for v in d.values()))
    print(f"{name:5s}: {len(d):4d} claims | {counts}")

In [ ]:
ft_model, ft_tokenizer = train_deberta(
    train_data            = train_pipeline,   # pipeline-retrieved evidence
    dev_data              = dev_pipeline,     # pipeline-retrieved evidence
    evidence_dict         = evidence_dict,
    bm25_retriever        = None,             # NEI uses cached IDs, not runtime BM25
    deberta_best_dir      = DEBERTA_BEST_DIR,
    hf_deberta_repo       = HF_DEBERTA_REPO,
    epochs                = 5,
    batch_size            = 2,
    disputed_weight_boost = 1.5,
    reuse_if_found        = False,            # force retrain; old checkpoint was trained on corrupt labels
)

In [ ]:
# -- Generate test-set predictions for submission -----------------------------
# The test set (test-claims-unlabelled.json) has no gold labels, so macro-F1 is
# impossible. Instead, run the trained model over the retrieved evidence and
# write a submission JSON: {claim_id: {claim_text, claim_label, evidences}}.
import json
import torch
from collections import Counter

ft_model.eval()
id2label = {int(k): v for k, v in ft_model.config.id2label.items()}

predictions = {}
items = list(test_pipeline.items())
BATCH = 16
for i in range(0, len(items), BATCH):
    batch    = items[i:i + BATCH]
    claims   = [v["claim_text"] for _, v in batch]
    ev_texts = [" ".join(evidence_dict.get(ev, "") for ev in v["evidences"][:3])
                for _, v in batch]
    enc = ft_tokenizer(ev_texts, claims, max_length=256,
                       truncation=True, padding=True, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        pred_ids = ft_model(**enc).logits.argmax(dim=-1).cpu().tolist()
    for (cid, v), pid in zip(batch, pred_ids):
        predictions[cid] = {
            "claim_text":  v["claim_text"],
            "claim_label": id2label[pid],
            "evidences":   v["evidences"],
        }

print(f"Predicted {len(predictions)} test claims | "
      f"{dict(Counter(p['claim_label'] for p in predictions.values()))}")

OUT_PATH = os.path.join(BASE_DIR, "outputs", "test-predictions-pipeline-deberta.json")
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w") as f:
    json.dump(predictions, f, indent=2)
print(f"Saved -> {OUT_PATH}")